In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: employee_src
position:
  x: 0
  y: 0
previewCodeHash: 030a9ce9cdfbe3f5
previewMode: "1000"
config:
  table_source:
    tableName: workspace.`default`.employees
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "false")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.`default`.employees"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_0.data"])

In [0]:
"""
id: select_5
template: transform
templateVersion: 2.0.0
name: select_columns
position:
  x: 228
  y: 1.0000038146972656
description:
  text: Rename selected columns and keep all columns in original order.
  hash: 0cf1af7f
previewCodeHash: 7f4de58c1e7435c3
previewMode: "1000"
config:
  mode: passthrough
  edits:
    - column: First Name
      alias: First_Name
    - column: Gender
      alias: Gender
    - column: Start Date
      alias: Start_date
    - column: Last Login Time
      alias: Last_login_time
    - column: Salary
      alias: Salary
    - column: Bonus %
      alias: Bonus
    - column: Senior Management
      alias: Sr_mgt
    - column: Team
      alias: Team
  ordered: []
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List
from pyspark.sql import functions as F

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "false")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "passthrough",
    "edits": [
        {
            "column": "First Name",
            "alias": "First_Name"
        },
        {
            "column": "Gender",
            "alias": "Gender"
        },
        {
            "column": "Start Date",
            "alias": "Start_date"
        },
        {
            "column": "Last Login Time",
            "alias": "Last_login_time"
        },
        {
            "column": "Salary",
            "alias": "Salary"
        },
        {
            "column": "Bonus %",
            "alias": "Bonus"
        },
        {
            "column": "Senior Management",
            "alias": "Sr_mgt"
        },
        {
            "column": "Team",
            "alias": "Team"
        }
    ],
    "ordered": []
}
inputs = {
    "data": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["select_5.transformed_data"] = out["transformed_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["select_5.transformed_data"])

In [0]:
"""
id: filter_1
template: filter
templateVersion: 2.0.0
name: filter_male
position:
  x: 453
  y: -5
description:
  text: Keep only rows where gender is male and separate the rest.
  hash: e73fa394
previewCodeHash: 49ea298183f5857f
previewMode: "1000"
config:
  condition: Gender = 'Male'
input:
  - node: select_5
    input_port: data
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "false")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "Gender = 'Male'"
}
inputs = {
    "data": ctx["select_5.transformed_data"]
}
out = run(config, inputs, spark)
ctx["filter_1.filtered_data"] = out["filtered_data"]
ctx["filter_1.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["filter_1.filtered_data"])
    display(ctx["filter_1.excluded_data"])

In [0]:
"""
id: limit_3
template: limit
templateVersion: 1.0.0
name: limit_dataset
position:
  x: 719.1551915376907
  y: -15.612550720736028
description:
  text: Limit the data to the first 100 rows.
  hash: 247af9d7
previewCodeHash: 88d28937db6b208f
previewMode: "1000"
config:
  limit: "100"
input:
  - node: filter_1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    n = int(config.get("limit", 1000))

    return {"limited_data": df.limit(n)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "false")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "limit": "100"
}
inputs = {
    "data": ctx["filter_1.filtered_data"]
}
out = run(config, inputs, spark)
ctx["limit_3.limited_data"] = out["limited_data"]
if globals().get("ld_display_outputs", False):
    display(ctx["limit_3.limited_data"])

In [0]:
"""
id: output_4
template: output
templateVersion: 2.0.0
name: store_data
position:
  x: 1075.3223728170742
  y: 54.298539310653034
description:
  text: Overwrite table empoyee_res in workspace.default schema with new data.
  hash: a514b227
previewMode: "1000"
config:
  output_type: table
  catalog: workspace
  schema: "`default`"
  table_name: empoyee_res
  write_mode: overwrite
input:
  - node: limit_3
    input_port: data
    output_port: limited_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    "'delta.columnMapping.mode' = 'name'"
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _tbl_properties_clause(user_props: str, format_value: str) -> str:
    parts: List[str] = []
    if format_value == "uniform":
        parts.append(UNIFORM_PROPS)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json"}

MAX_SINGLE_FILE_ROWS = 1_000_000

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{file_type}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import uuid

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        tmp_path = f"{path}.{uuid.uuid4().hex}.tmp"
        try:
            write_body(tmp_path)
        except BaseException:
            _remove(tmp_path)
            raise
        _remove(path)
        os.rename(tmp_path, path)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and set(header) != set(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([row[c] for c in header])

        _stage(_write_csv)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv or json."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(table_properties, format_value)

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "`default`",
    "table_name": "empoyee_res",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["limit_3.limited_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: visualization_6
template: visualization
templateVersion: 1.0.0
name: visualization_data
position:
  x: 1027.3958228588494
  y: -86.12906185177097
description:
  text: Count entries for each team.
  hash: 53778fe4
previewCodeHash: 5dcd0748a5891f51
previewMode: "1000"
config:
  editorSpec:
    queries:
      - queryName: main_query
        query:
          datasetName: lakeflow_designer_upstream
          fields:
            - expression: COUNT(`*`)
              fieldName: count(*)
            - expression: "`Team`"
              fieldName: Team
          disaggregatedData: false
    renderSpec:
      version: 3
      widgetType: pie
      encodings:
        angle:
          fieldName: count(*)
          scale:
            type: quantitative
        color:
          fieldName: Team
          scale:
            type: categorical
        label:
          show: true
      data:
        queryName: main_query
    parameterQueries: []
    version: 2
  sql_query: |-
    WITH q AS (SELECT * FROM lakeflow_designer_upstream)
    SELECT COUNT(*) AS `count(*)`, `Team` AS Team
    FROM q
    GROUP BY Team
input:
  - node: limit_3
    input_port: data
    output_port: limited_data
"""

# generated from the system
from typing import Any, Dict

_SOURCE_VIEW = "lakeflow_designer_upstream"

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        return {"data": None}

    sql_query = config.get("sql_query")
    if not isinstance(sql_query, str) or not sql_query.strip():
        return {"data": df}

    try:
        df.createOrReplaceTempView(_SOURCE_VIEW)
        globals().setdefault("_lb_views", set()).add(_SOURCE_VIEW)
        return {"data": spark.sql(sql_query)}
    except Exception as exc:
        print("[visualization] compiled query failed, passing input through:", exc, sql_query)
        return {"data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "false")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "editorSpec": {
        "queries": [
            {
                "queryName": "main_query",
                "query": {
                    "datasetName": "lakeflow_designer_upstream",
                    "fields": [
                        {
                            "expression": "COUNT(`*`)",
                            "fieldName": "count(*)"
                        },
                        {
                            "expression": "`Team`",
                            "fieldName": "Team"
                        }
                    ],
                    "disaggregatedData": False
                }
            }
        ],
        "renderSpec": {
            "facet": None,
            "version": 3,
            "widgetType": "pie",
            "encodings": {
                "angle": {
                    "fieldName": "count(*)",
                    "scale": {
                        "type": "quantitative"
                    }
                },
                "color": {
                    "fieldName": "Team",
                    "scale": {
                        "type": "categorical"
                    }
                },
                "label": {
                    "show": True
                }
            },
            "data": {
                "queryName": "main_query"
            }
        },
        "parameterQueries": [],
        "version": 2
    },
    "sql_query": "WITH q AS (SELECT * FROM lakeflow_designer_upstream)\nSELECT COUNT(*) AS `count(*)`, `Team` AS Team\nFROM q\nGROUP BY Team"
}
inputs = {
    "data": ctx["limit_3.limited_data"]
}
out = run(config, inputs, spark)
ctx["visualization_6.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["visualization_6.data"])